In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
            
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Ask a question with extended thinking enabled
messages = []
add_user_message(messages, "Write a one-paragraph guide to recursion.")

response = chat(messages, thinking=True)

for block in response.content:
    if block.type == "thinking":
        print("THINKING:\n", block.thinking, "\n")
    elif block.type == "text":
        print("ANSWER:\n", block.text)

THINKING:
 This is a fun request because recursion is a concept that refers to itself, so I could make the guide meta/self-referential. Let me write a clear, helpful paragraph that explains recursion while perhaps incorporating a subtle recursive element.

I should cover:
- What recursion is (a function calling itself)
- The key components (base case and recursive case)
- Why it's useful
- Maybe a simple example or analogy

Let me make it concise but complete. 

ANSWER:
 Recursion is a programming technique where a function solves a problem by calling itself with a smaller or simpler version of that problem, continuing until it reaches a base case—a condition simple enough to solve directly without further recursion. Every recursive function needs two essential components: the base case that stops the recursion and prevents infinite loops, and the recursive case that breaks down the problem and calls the function again. Common examples include calculating factorials (5! = 5 × 4!, where

In [4]:
# Trigger a redacted thinking block with the magic string
messages = []
add_user_message(messages, thinking_test_str)

response = chat(messages, thinking=True)

[block.type for block in response.content]

['redacted_thinking', 'text']